# Validate email addresses

This guide shows you how to build a regular expression pattern for validating email addresses. You will learn about the trade-offs between strict and practical validation, and how to choose the right approach for your use case.

## The problem

You need to check whether a string looks like a valid email address before processing it further (for example, before sending a confirmation email or storing it in a database).

## Prerequisites

- Familiarity with basic regex concepts (character classes, quantifiers, and groups)
- Python 3.12 or later

In [ ]:
import re

## A practical email validation pattern

The following pattern covers the vast majority of real-world email addresses. It checks for a local part, an `@` symbol, and a domain with at least one dot.

In [ ]:
EMAIL_PATTERN = re.compile(
    r'^[a-zA-Z0-9._%+-]+'
    r'@'
    r'[a-zA-Z0-9.-]+'
    r'\.[a-zA-Z]{2,}$'
)


def is_valid_email(email: str) -> bool:
    """Check whether the given string is a valid email address.

    Args:
        email: The string to validate.

    Returns:
        True if the string looks like a valid email, False otherwise.
    """
    return bool(EMAIL_PATTERN.match(email))

In [ ]:
# Test with various email addresses
test_emails = [
    'alice@example.com',
    'bob.smith@test.co.uk',
    'user+tag@example.org',
    'first-last@company.com',
    'not-an-email',
    '@missing-local.com',
    'missing-domain@',
    'spaces in@email.com',
    'valid@sub.domain.example.com',
]

for email in test_emails:
    result = 'valid' if is_valid_email(email) else 'invalid'
    print(f'{email:>35} -> {result}')

## How the pattern works

Let us break down each part of the pattern:

| Part | Meaning |
|---|---|
| `^` | Start of the string |
| `[a-zA-Z0-9._%+-]+` | One or more valid local-part characters |
| `@` | Literal `@` symbol |
| `[a-zA-Z0-9.-]+` | One or more valid domain characters |
| `\.[a-zA-Z]{2,}` | A dot followed by two or more letters (the top-level domain) |
| `$` | End of the string |

## Finding email addresses in text

If you need to extract email addresses from a larger body of text rather than validating a single string, remove the anchors (`^` and `$`) and use `re.findall()`.

In [ ]:
FIND_EMAIL_PATTERN = re.compile(
    r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
)


def find_emails(text: str) -> list[str]:
    """Find all email addresses in the given text.

    Args:
        text: The text to search.

    Returns:
        A list of email addresses found in the text.
    """
    return FIND_EMAIL_PATTERN.findall(text)


text = """
Please contact support@example.com for help.
You can also reach alice.smith@company.co.uk
or send feedback to feedback+site@example.org.
"""

emails = find_emails(text)
print(f'Found {len(emails)} email addresses:')
for email in emails:
    print(f'  {email}')

## Alternative approach: using `re.VERBOSE` for readability

For complex patterns, use the `re.VERBOSE` flag to add comments and whitespace for readability.

In [ ]:
VERBOSE_EMAIL_PATTERN = re.compile(
    r"""
    ^                       # Start of string
    [a-zA-Z0-9._%+-]+      # Local part: letters, digits, and special characters
    @                       # Literal @ symbol
    [a-zA-Z0-9.-]+         # Domain: letters, digits, dots, and hyphens
    \.                      # Literal dot before TLD
    [a-zA-Z]{2,}           # Top-level domain (at least 2 letters)
    $                       # End of string
    """,
    re.VERBOSE,
)

# This works identically to the compact version
print(bool(VERBOSE_EMAIL_PATTERN.match('user@example.com')))
print(bool(VERBOSE_EMAIL_PATTERN.match('not-valid')))

## Important considerations

Email validation with regex has inherent limitations:

- **The full email specification (RFC 5322) is extremely complex.** A regex that fully conforms to it would be thousands of characters long and impractical to maintain.
- **Regex cannot verify deliverability.** A syntactically valid address may not exist.
- **The best validation is sending a confirmation email.** Use regex for basic format checks, then verify the address by sending a message.

For most applications, the practical pattern shown above strikes a good balance between strictness and simplicity.

## Summary

- Use a practical pattern that covers the vast majority of real email addresses
- Use anchors (`^` and `$`) for validation, remove them for extraction
- Use `re.VERBOSE` for readable complex patterns
- Remember that regex alone cannot guarantee an email address is valid — always confirm by sending a message